In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as pl

In [4]:
df = pd.read_csv('qoute_dataset.csv')

In [5]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [6]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [7]:
df.shape

(3038, 2)

In [8]:
quotes = df['quote']

In [9]:
quotes = quotes.str.lower()

In [10]:
import string
translator = str.maketrans('','',string.punctuation)
quotes = quotes.apply(lambda x : x.translate(translator))

In [11]:
quotes[0]

'“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”'

In [12]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [13]:
vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [14]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [15]:
sequences = tokenizer.texts_to_sequences(quotes)

In [16]:
quotes[0]

'“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”'

In [17]:
sequences[0]

[713,
 62,
 29,
 19,
 16,
 946,
 10,
 7,
 5,
 1156,
 8,
 70,
 293,
 10,
 145,
 12,
 809,
 104,
 752,
 70,
 2461]

In [18]:
X = []
Y = []

for seq in sequences:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    Y.append(output_seq)


In [19]:
len(X)

85271

In [20]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Find the maximum length of sequences in X
maxlen = max(len(seq) for seq in X)
print(f"Maximum sequence length: {maxlen}")

# Apply padding to X
X_padded = pad_sequences(X, maxlen=maxlen, padding='pre') # 'pre' adds padding at the beginning

print(f"Shape of padded X: {X_padded.shape}")
print(f"Example of a padded sequence (first element):\n{X_padded[0]}")


Maximum sequence length: 745
Shape of padded X: (85271, 745)
Example of a padded sequence (first element):
[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0 

In [21]:
Y = np.array(Y)

In [22]:
from tensorflow.keras.utils import to_categorical

In [23]:
Y_one_hot = to_categorical(Y, num_classes=vocab_size + 1)

print(f"Shape of one-hot encoded Y: {Y_one_hot.shape}")
print(f"Example of one-hot encoded Y (first element):\n{Y_one_hot[0][:10]}") # Display first 10 elements for brevity

Shape of one-hot encoded Y: (85271, 10001)
Example of one-hot encoded Y (first element):
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [24]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional ,SimpleRNN

In [25]:
# Define model parameters
embed_dim = 50  # Dimension of the embedding vector
rnn_units = 128 # Number of units in the RNN layer

# Build the model
rnn_model = Sequential([
    Embedding(input_dim=vocab_size + 1, output_dim=embed_dim, input_length=maxlen),
    Bidirectional(SimpleRNN(rnn_units)),
    Dense(vocab_size + 1, activation='softmax')
])

# Compile the model
rnn_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Display model summary
rnn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [26]:
# Define model parameters for the new LSTM model
embed_dim_lstm = 50  # Dimension of the embedding vector
lstm_units_new = 128 # Number of units in the LSTM layer

# Build the new LSTM model
lstm_model = Sequential([
    Embedding(input_dim=vocab_size + 1, output_dim=embed_dim_lstm, input_length=maxlen),
    Bidirectional(LSTM(lstm_units_new)),
    Dense(vocab_size + 1, activation='softmax')
])

# Compile the new LSTM model
lstm_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Display new LSTM model summary
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [27]:
epochs = 100
batch_size = 128

In [28]:
history_lstm = lstm_model.fit(
    X_padded,
    Y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 63s 94ms/step - accuracy: 0.0428 - loss: 6.7194 - val_accuracy: 0.0555 - val_loss: 6.6315
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 80s 96ms/step - accuracy: 0.0678 - loss: 6.2307 - val_accuracy: 0.0776 - val_loss: 6.4761
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 82s 96ms/step - accuracy: 0.0904 - loss: 5.9313 - val_accuracy: 0.0937 - val_loss: 6.4101
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 58s 96ms/step - accuracy: 0.1063 - loss: 5.6711 - val_accuracy: 0.1015 - val_loss: 6.3775
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 58s 96ms/step - accuracy: 0.1185 - loss: 5.4375 - val_accuracy: 0.1046 - val_loss: 6.3724
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 58s 96ms/step - accuracy: 0.1288 - loss: 5.2298 - val_accuracy: 0.1075 - val_loss: 6.4129
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 58s 96ms/step - accuracy: 0.1385 - loss: 5.0383 - val_accuracy: 0.1134 - val_loss: 6.4263
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 58s 97ms/step - accuracy: 0.1475 - loss: 4

In [29]:
lstm_model.save('lstm_model.h5')

In [30]:
index_to_word = {}
for word, index in word_index.items():
    index_to_word[index] = word


In [31]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [34]:
def predictor(model,tokenizer,text,maxlen):
  text = text.lower()
  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq],maxlen=maxlen,padding='pre')

  pred = model.predict(seq,verbose=0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [39]:
seed_text ='i'
next_word = predictor(lstm_model,tokenizer,seed_text,maxlen)
print(f"The next word is: {next_word}")

The next word is: am


In [52]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [53]:
seed = "are you a "
generate_text = generate_text(lstm_model,tokenizer,seed,maxlen,10)
print(generate_text)

are you a  part of a small years the book is a part


In [54]:
import pickle

In [56]:
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokenizer, f)

In [58]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(maxlen, f)